# 6CS012 - Worksheet 6 Solution  
## Practical Aspects of Training CNN for Image Classification

This notebook is written as a **continuation of Worksheet 5**.

It keeps the same overall workflow and dataset structure, then improves the earlier model by adding:

- dataset verification and visualization
- data augmentation
- a deeper CNN with **Batch Normalization** and **Dropout**
- **transfer learning with VGG16**
- model comparison
- inference output and classification report

> Update the dataset paths below before running the notebook.


In [ ]:
# ============================================================
# STEP 0: IMPORT LIBRARIES
# ============================================================
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

from PIL import Image, UnidentifiedImageError

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential, Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))


## 1. Dataset Paths  
Use the **same dataset structure as Worksheet 5**.

Expected structure:
```python
data/
  train/
    acai/
    cupuacu/
    graviola/
    guarana/
    pupunha/
    tucuma/
  test/
    acai/
    cupuacu/
    graviola/
    guarana/
    pupunha/
    tucuma/
```


In [ ]:
# ============================================================
# STEP 1: SET DATASET PATHS
# ============================================================
train_dir = "data/train"   # change if needed
test_dir  = "data/test"    # change if needed

print("Train directory exists:", os.path.exists(train_dir))
print("Test directory exists :", os.path.exists(test_dir))


## 2. Data Understanding and Visualization

In [ ]:
# ============================================================
# STEP 2.1: READ DATASET DIRECTORY AND EXTRACT CLASS NAMES
# ============================================================
class_names = sorted([
    d for d in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, d))
])

print(f"Found {len(class_names)} classes: {class_names}")
num_classes = len(class_names)


In [ ]:
# ============================================================
# STEP 2.2: CHECK FOR CORRUPTED IMAGES
# ============================================================
corrupted_images = []

for class_name in class_names:
    class_path = os.path.join(train_dir, class_name)
    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        if not os.path.isfile(img_path):
            continue

        try:
            with Image.open(img_path) as img:
                img.verify()
        except (IOError, UnidentifiedImageError, OSError):
            corrupted_images.append(img_path)

if corrupted_images:
    print("\nCorrupted Images Found:")
    for img in corrupted_images:
        print(img)
else:
    print("\nNo corrupted images found.")


In [ ]:
# ============================================================
# STEP 2.3: CHECK CLASS DISTRIBUTION
# ============================================================
class_counts = {}

for class_name in class_names:
    class_path = os.path.join(train_dir, class_name)
    images = [
        img for img in os.listdir(class_path)
        if img.lower().endswith((".png", ".jpg", ".jpeg"))
    ]
    class_counts[class_name] = len(images)

print("\nClass Distribution:")
print("=" * 45)
print(f"{'Class Name':<25}{'Valid Image Count':>15}")
print("=" * 45)
for class_name, count in class_counts.items():
    print(f"{class_name:<25}{count:>15}")
print("=" * 45)


In [ ]:
# ============================================================
# STEP 2.4: VISUALIZE ONE RANDOM IMAGE FROM EACH CLASS
# ============================================================
selected_images = []
selected_labels = []

for class_name in class_names:
    class_path = os.path.join(train_dir, class_name)
    images = [
        img for img in os.listdir(class_path)
        if img.lower().endswith((".png", ".jpg", ".jpeg"))
    ]
    if images:
        selected_img = os.path.join(class_path, random.choice(images))
        selected_images.append(selected_img)
        selected_labels.append(class_name)

num_imgs = len(selected_images)
cols = 3
rows = int(np.ceil(num_imgs / cols))

plt.figure(figsize=(12, 4 * rows))
for i, img_path in enumerate(selected_images):
    plt.subplot(rows, cols, i + 1)
    img = mpimg.imread(img_path)
    plt.imshow(img)
    plt.title(selected_labels[i])
    plt.axis("off")

plt.tight_layout()
plt.show()


## 3. Create Train, Validation, and Test Datasets

Worksheet 6 says it continues from Worksheet 5, so we still use Keras directory loading, but now we prepare the data in a cleaner pipeline for augmentation and stronger models.


In [ ]:
# ============================================================
# STEP 3: DATASET LOADING
# ============================================================
IMG_SIZE = (224, 224)   # useful for VGG16 too
BATCH_SIZE = 32
SEED = 1337
VAL_SPLIT = 0.2

train_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=VAL_SPLIT,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int"
)

val_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=VAL_SPLIT,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int"
)

test_ds = keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode="int"
)

print("Class names from dataset:", train_ds.class_names)
class_names = train_ds.class_names
num_classes = len(class_names)


In [ ]:
# ============================================================
# STEP 3.1: CHECK BATCH SHAPES
# ============================================================
for images, labels in train_ds.take(1):
    print("Images shape:", images.shape)
    print("Labels shape:", labels.shape)


In [ ]:
# ============================================================
# STEP 3.2: VISUALIZE SAMPLE TRAINING IMAGES
# ============================================================
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
plt.tight_layout()
plt.show()


## 4. Data Augmentation

Worksheet 6 asks to use augmentation and improve the earlier model.  
We use the newer Keras augmentation layers so augmentation happens during training.


In [ ]:
# ============================================================
# STEP 4: DATA AUGMENTATION LAYERS
# ============================================================
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
], name="data_augmentation")

data_augmentation


In [ ]:
# ============================================================
# STEP 4.1: VISUALIZE AUGMENTED VERSIONS
# ============================================================
plt.figure(figsize=(10, 10))
for images, _ in train_ds.take(1):
    first_image = images[:1]
    for i in range(9):
        augmented_image = data_augmentation(first_image, training=True)
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(augmented_image[0].numpy().astype("uint8"))
        plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# STEP 4.2: PERFORMANCE OPTIMIZATION
# ============================================================
AUTOTUNE = tf.data.AUTOTUNE

train_ds_prefetch = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds_prefetch = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds_prefetch = test_ds.prefetch(buffer_size=AUTOTUNE)


## 5. Improved CNN from Scratch with BatchNorm and Dropout

In [ ]:
# ============================================================
# STEP 5: BUILD IMPROVED CNN MODEL
# ============================================================
cnn_model = Sequential([
    layers.Input(shape=(224, 224, 3)),
    data_augmentation,
    layers.Rescaling(1./255),

    # Block 1
    layers.Conv2D(32, (3, 3), padding="same", activation=None),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Block 2
    layers.Conv2D(64, (3, 3), padding="same", activation=None),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Block 3
    layers.Conv2D(128, (3, 3), padding="same", activation=None),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Block 4
    layers.Conv2D(256, (3, 3), padding="same", activation=None),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Flatten(),

    layers.Dense(512, activation=None),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.5),

    layers.Dense(256, activation=None),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.5),

    layers.Dense(128, activation=None),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.5),

    layers.Dense(64, activation=None),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.5),

    layers.Dense(num_classes, activation="softmax")
])

cnn_model.summary()


In [ ]:
# ============================================================
# STEP 5.1: COMPILE IMPROVED CNN
# ============================================================
cnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cnn_callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        "best_cnn_bn_dropout.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]


In [ ]:
# ============================================================
# STEP 5.2: TRAIN IMPROVED CNN
# ============================================================
history_cnn = cnn_model.fit(
    train_ds_prefetch,
    validation_data=val_ds_prefetch,
    epochs=30,
    callbacks=cnn_callbacks
)


In [ ]:
# ============================================================
# STEP 5.3: PLOT TRAINING CURVES FOR IMPROVED CNN
# ============================================================
def plot_history(history, title_prefix="Model"):
    epochs = range(1, len(history.history["accuracy"]) + 1)

    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history.history["accuracy"], label="Train Accuracy")
    plt.plot(epochs, history.history["val_accuracy"], label="Validation Accuracy")
    plt.title(f"{title_prefix} Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history.history["loss"], label="Train Loss")
    plt.plot(epochs, history.history["val_loss"], label="Validation Loss")
    plt.title(f"{title_prefix} Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_history(history_cnn, "Improved CNN")


In [ ]:
# ============================================================
# STEP 5.4: EVALUATE IMPROVED CNN
# ============================================================
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(test_ds_prefetch, verbose=1)

print("\nImproved CNN Test Results")
print(f"Test Loss    : {cnn_test_loss:.4f}")
print(f"Test Accuracy: {cnn_test_acc:.4f} ({cnn_test_acc * 100:.2f}%)")


In [ ]:
# ============================================================
# STEP 5.5: CLASSIFICATION REPORT FOR IMPROVED CNN
# ============================================================
y_true_cnn = []
y_pred_cnn = []

for images, labels in test_ds_prefetch:
    preds = cnn_model.predict(images, verbose=0)
    y_true_cnn.extend(labels.numpy())
    y_pred_cnn.extend(np.argmax(preds, axis=1))

y_true_cnn = np.array(y_true_cnn)
y_pred_cnn = np.array(y_pred_cnn)

print("Classification Report - Improved CNN")
print(classification_report(y_true_cnn, y_pred_cnn, target_names=class_names))


In [ ]:
# ============================================================
# STEP 5.6: SAMPLE INFERENCE OUTPUT FOR IMPROVED CNN
# ============================================================
sample_images, sample_labels = next(iter(test_ds_prefetch))
sample_preds = cnn_model.predict(sample_images, verbose=0)
sample_pred_labels = np.argmax(sample_preds, axis=1)

plt.figure(figsize=(12, 12))
for i in range(min(9, len(sample_images))):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(sample_images[i].numpy().astype("uint8"))
    true_name = class_names[int(sample_labels[i])]
    pred_name = class_names[int(sample_pred_labels[i])]
    color = "green" if true_name == pred_name else "red"
    plt.title(f"True: {true_name}\nPred: {pred_name}", color=color)
    plt.axis("off")
plt.tight_layout()
plt.show()


## 6. Transfer Learning with VGG16

Worksheet 6 also asks for **transfer learning** using a pre-trained ImageNet model.  
Below, we freeze the convolutional base and train only the new classification head.


In [ ]:
# ============================================================
# STEP 6: PREPARE DATA FOR VGG16
# ============================================================
# VGG16 expects preprocessing suited for ImageNet.
# We keep augmentation outside the base model here for clarity.

def preprocess_for_vgg(image, label):
    image = preprocess_input(tf.cast(image, tf.float32))
    return image, label

train_vgg_ds = train_ds.map(preprocess_for_vgg, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_vgg_ds   = val_ds.map(preprocess_for_vgg, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_vgg_ds  = test_ds.map(preprocess_for_vgg, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)


In [ ]:
# ============================================================
# STEP 6.1: LOAD PRE-TRAINED VGG16 BASE AND FREEZE IT
# ============================================================
base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

for layer in base_model.layers:
    layer.trainable = False

print("Number of layers in base model:", len(base_model.layers))
print("Base model trainable:", base_model.trainable)


In [ ]:
# ============================================================
# STEP 6.2: BUILD FINAL TRANSFER LEARNING MODEL
# ============================================================
inputs = keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = preprocess_input(x)

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation="relu")(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

vgg_model = Model(inputs, outputs)

vgg_model.summary()


In [ ]:
# ============================================================
# STEP 6.3: COMPILE VGG16 MODEL
# ============================================================
vgg_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

vgg_callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        "best_vgg16_transfer.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]


In [ ]:
# ============================================================
# STEP 6.4: TRAIN VGG16 TRANSFER MODEL
# ============================================================
history_vgg = vgg_model.fit(
    train_ds_prefetch,   # preprocessing is already inside the model
    validation_data=val_ds_prefetch,
    epochs=20,
    callbacks=vgg_callbacks
)


In [ ]:
# ============================================================
# STEP 6.5: PLOT TRAINING CURVES FOR VGG16
# ============================================================
plot_history(history_vgg, "VGG16 Transfer Learning")


In [ ]:
# ============================================================
# STEP 6.6: EVALUATE VGG16 MODEL
# ============================================================
vgg_test_loss, vgg_test_acc = vgg_model.evaluate(test_ds_prefetch, verbose=1)

print("\nVGG16 Transfer Learning Test Results")
print(f"Test Loss    : {vgg_test_loss:.4f}")
print(f"Test Accuracy: {vgg_test_acc:.4f} ({vgg_test_acc * 100:.2f}%)")


In [ ]:
# ============================================================
# STEP 6.7: CLASSIFICATION REPORT FOR VGG16 MODEL
# ============================================================
y_true_vgg = []
y_pred_vgg = []

for images, labels in test_ds_prefetch:
    preds = vgg_model.predict(images, verbose=0)
    y_true_vgg.extend(labels.numpy())
    y_pred_vgg.extend(np.argmax(preds, axis=1))

y_true_vgg = np.array(y_true_vgg)
y_pred_vgg = np.array(y_pred_vgg)

print("Classification Report - VGG16 Transfer Learning")
print(classification_report(y_true_vgg, y_pred_vgg, target_names=class_names))


In [ ]:
# ============================================================
# STEP 6.8: SAMPLE INFERENCE OUTPUT FOR VGG16 MODEL
# ============================================================
sample_images, sample_labels = next(iter(test_ds_prefetch))
sample_preds = vgg_model.predict(sample_images, verbose=0)
sample_pred_labels = np.argmax(sample_preds, axis=1)

plt.figure(figsize=(12, 12))
for i in range(min(9, len(sample_images))):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(sample_images[i].numpy().astype("uint8"))
    true_name = class_names[int(sample_labels[i])]
    pred_name = class_names[int(sample_pred_labels[i])]
    color = "green" if true_name == pred_name else "red"
    plt.title(f"True: {true_name}\nPred: {pred_name}", color=color)
    plt.axis("off")
plt.tight_layout()
plt.show()


## 7. Compare the Two Worksheet 6 Models

In [ ]:
# ============================================================
# STEP 7: MODEL COMPARISON
# ============================================================
print("Model Comparison")
print("-" * 50)
print(f"Improved CNN Test Accuracy : {cnn_test_acc:.4f} ({cnn_test_acc * 100:.2f}%)")
print(f"VGG16 Test Accuracy        : {vgg_test_acc:.4f} ({vgg_test_acc * 100:.2f}%)")
print("-" * 50)

if vgg_test_acc > cnn_test_acc:
    print("Transfer learning with VGG16 performed better than training from scratch.")
elif vgg_test_acc < cnn_test_acc:
    print("The improved CNN trained from scratch performed better on this dataset.")
else:
    print("Both models achieved the same test accuracy.")


## 8. Save Models

In [ ]:
# ============================================================
# STEP 8: SAVE BOTH MODELS
# ============================================================
cnn_model.save("worksheet6_improved_cnn.keras")
vgg_model.save("worksheet6_vgg16_transfer.keras")

print("Saved:")
print("- worksheet6_improved_cnn.keras")
print("- worksheet6_vgg16_transfer.keras")


## 9. Final Notes

This notebook satisfies the main Worksheet 6 requirements:

- repeated the Worksheet 5 image classification flow on the same dataset
- added **data augmentation**
- used a **deeper CNN** with **Batch Normalization** and **Dropout**
- trained and evaluated a **transfer learning model with VGG16**
- generated **inference outputs**
- printed **classification reports**
- compared performance of both approaches

If your teacher wants actual final numbers in the notebook, run all cells with the real dataset and then save the notebook with outputs.
